# 🏷️ Week 6: Part-of-Speech (POS) Tagging Bahasa Indonesia

## Tujuan Pembelajaran:
1. Memahami konsep **POS Tagging** — proses pemberian label kelas kata pada setiap token (kata benda, kata kerja, kata sifat, dll.)
2. Menggunakan **Polyglot** untuk POS tagging berbahasa Indonesia
3. Menerapkan deteksi bahasa (*language detection*) sebelum melakukan tagging
4. Menganalisis distribusi POS tag dari seluruh dataset ulasan Honest Review
5. Menemukan kata (token) yang paling sering muncul berdasarkan kelas katanya

---
> 📂 **Dataset**: Honest Review (`cleandata.csv`) — kolom `text_final` (teks yang sudah dipreproses/di-stem)
> File ini berada di folder `Week 3/` dalam workspace yang sama.
>
> ℹ️ **Polyglot** memerlukan beberapa dependensi tambahan di sistem (ICU, PyICU). Jalankan sel instalasi di bawah dan ikuti instruksinya.

## 1) Install Library

Polyglot memerlukan beberapa library native. Jalankan perintah berikut sesuai dengan environment yang digunakan.

In [ ]:
!pip install polyglot pycld2 morfessor pandas matplotlib
# Jika di Linux/Colab, jalankan juga:
# !apt-get install -y libicu-dev
# !pip install pyicu

In [ ]:
# Download data Polyglot untuk bahasa Indonesia
!polyglot download embeddings2.id
!polyglot download pos2.id

## 2) Import Library

Import modul yang dibutuhkan untuk POS tagging dan deteksi bahasa.

In [ ]:
import pandas as pd
import unicodedata
import re
from collections import defaultdict

from polyglot.text import Text
from polyglot.detect import Detector

print("Library berhasil diimport!")

## 3) Load Dataset

Load dataset Honest Review dari `cleandata.csv`. Kolom `text_final` berisi teks yang sudah melalui proses tokenisasi, stopword removal, dan stemming.

In [ ]:
file_path = '../Week 3/cleandata.csv'
df_text = pd.read_csv(file_path)

print(f"Jumlah data: {len(df_text)}")
print(f"Kolom: {list(df_text.columns)}")
df_text['text_final'].head()

## 4) Pembersihan Teks

Polyglot memerlukan teks yang bebas dari karakter non-printable dan non-ASCII. Fungsi `clean_text()` memastikan teks kompatibel dengan library ini.

In [ ]:
def clean_text(text):
    """Bersihkan teks dari karakter non-printable dan non-ASCII."""
    text = str(text)
    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != 'C')
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text.strip()

# Terapkan ke seluruh dataset
df_text['text_clean'] = df_text['text_final'].apply(clean_text)
print("Pembersihan teks selesai!")
print("Contoh hasil:")
for i in range(3):
    print(f"  [{i}] {repr(df_text['text_clean'].iloc[i][:80])}")

## 5) Deteksi Bahasa

Sebelum melakukan POS tagging, deteksi bahasa teks untuk memastikan Polyglot menggunakan model yang tepat. Sampel ulasan pertama diperiksa apakah terdeteksi sebagai Bahasa Indonesia (`id`).

In [ ]:
def detect_language(text):
    """Deteksi bahasa menggunakan Polyglot. Return kode ISO bahasa."""
    try:
        return Detector(text, quiet=True).language.code
    except Exception:
        return 'unknown'

print("Deteksi bahasa untuk 10 sampel pertama:")
sample_langs = df_text['text_clean'].head(10).apply(detect_language)
for i, (txt, lang) in enumerate(zip(df_text['text_clean'].head(10), sample_langs)):
    print(f"  [{i}] lang={lang!r} | {repr(txt[:60])}")

## 6) POS Tagging — Demo Satu Kalimat

Demonstrasi POS tagging pada satu ulasan. Setiap token diberi label kelas kata menggunakan skema **Universal POS Tags (UPOS)**:

| Tag | Kelas Kata | Contoh |
|-----|-----------|--------|
| `NOUN` | Kata Benda | aplikasi, layanan |
| `VERB` | Kata Kerja | menggunakan, daftar |
| `ADJ`  | Kata Sifat | bagus, mudah |
| `ADV`  | Kata Keterangan | sangat, cepat |
| `PRON` | Kata Ganti | saya, mereka |
| `ADP`  | Preposisi | di, ke, dari |
| `DET`  | Determiner | ini, itu |
| `CONJ` | Konjungsi | dan, atau |
| `NUM`  | Angka | satu, dua |
| `PUNCT`| Tanda Baca | . , ! |

In [ ]:
sample_text = df_text['text_clean'].iloc[0]
print(f"Teks: {repr(sample_text[:100])}")
print()

polyglot_text = Text(sample_text, hint_language_code='id')
pos_result = polyglot_text.pos_tags

# Tampilkan dalam bentuk tabel
print(f"{'No.':<5} {'Token':<25} {'POS Tag'}")
print('-' * 45)
for i, (word, tag) in enumerate(pos_result):
    print(f"{i:<5} {word:<25} {tag}")

## 7) POS Tagging Seluruh Dataset

Terapkan POS tagging ke seluruh dataset. Proses ini dapat memakan beberapa menit tergantung jumlah data. Hasil disimpan sebagai list pasangan (word, tag) per ulasan.

In [ ]:
all_pos_tags = []

for i, row in df_text.iterrows():
    text = row['text_clean']
    if not text or len(text.split()) < 2:
        all_pos_tags.append([])
        continue
    try:
        lang = detect_language(text)
        if lang in ('id', 'un', 'unknown'):
            t = Text(text, hint_language_code='id')
        else:
            t = Text(text)
        all_pos_tags.append(t.pos_tags)
    except Exception:
        all_pos_tags.append([])

    if (i + 1) % 5000 == 0:
        print(f"  Diproses: {i+1}/{len(df_text)}")

print(f"\nSelesai! Total dokumen: {len(all_pos_tags)}")

## 8) Distribusi POS Tag

Gabungkan semua pasangan (word, tag) menjadi satu DataFrame dan analisis distribusi frekuensi setiap kelas kata.

In [ ]:
# Gabungkan semua (word, tag) ke satu list
flat_pos = [(word, tag) for doc in all_pos_tags for word, tag in doc]
df_word_pos = pd.DataFrame(flat_pos, columns=['Word', 'POS'])

print(f"Total pasangan (word, tag): {len(df_word_pos):,}")

# Hitung distribusi tag
tag_counts = df_word_pos['POS'].value_counts().reset_index()
tag_counts.columns = ['POS', 'Frekuensi']

unique_per_tag = df_word_pos.groupby('POS')['Word'].nunique().reset_index()
unique_per_tag.columns = ['POS', 'Unique_Tokens']

summary_df = tag_counts.merge(unique_per_tag, on='POS')
summary_df['Persentase'] = (summary_df['Frekuensi'] / summary_df['Frekuensi'].sum() * 100).round(2)
print("\n=== Distribusi POS Tag ===")
print(summary_df.to_string(index=False))

# Simpan ke CSV
summary_df.to_csv('pos_tags_summary.csv', index=False)
print("\nDisimpan ke pos_tags_summary.csv")

## 9) Visualisasi dan Top Token per Tag

Plot distribusi kelas kata dan tampilkan 10 token paling sering per POS tag.

In [ ]:
# Bar chart distribusi POS tag
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.bar(summary_df['POS'], summary_df['Frekuensi'], color='steelblue', edgecolor='white')
plt.xlabel('POS Tag', fontsize=12)
plt.ylabel('Frekuensi', fontsize=12)
plt.title('Distribusi POS Tag — Honest Review', fontsize=14)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 token per POS tag
top_entities = (
    df_word_pos.groupby(['POS', 'Word'])
    .size().reset_index(name='Count')
    .sort_values(['POS', 'Count'], ascending=[True, False])
)
top_entities = top_entities.groupby('POS').head(10)
top_entities.to_csv('pos_top_tokens.csv', index=False)

print("=== Top 10 Token per POS Tag ===")
for tag in top_entities['POS'].unique():
    subset = top_entities[top_entities['POS'] == tag].head(5)
    words = ', '.join(f"{r['Word']} ({r['Count']})" for _, r in subset.iterrows())
    print(f"  {tag:<8}: {words}")

print("\nDisimpan ke pos_top_tokens.csv")